# Row 2: the decider with the cues in its prompt

What this tests and why: `docs/RATIONALE.md` (section 4, row 2; decision D21). Same model and same question as row 1, with one change:
the three cues from row 3 are written into the prompt. Everything else is held fixed, so any difference is due to the cues.

It reuses the cues already written (row 3's cue file). Only the decider runs: about 10 minutes for the regular 2 s ticks on val, and about
8 more for the moments where the matcher fires (so row 4's combination can be redone with the better-informed decider).

In [ ]:
import os
if os.path.basename(os.getcwd()) == "notebooks":      # run from the repo root
    os.chdir("..")
import glob, torch
print(os.getcwd(), "| cuda:", torch.cuda.is_available())
CUES    = sorted(glob.glob("results/cache/cues/*/cues.jsonl"))[-1]
MATCHER = sorted(glob.glob("results/cache/matcher/Ops-MM-embedding-v1-2B__*"))[-1]
ROW1    = sorted(glob.glob("results/cache/decider/Qwen3-VL-2B-Instruct__query_only__*"))[-1]
print("cues:", CUES); print("matcher:", MATCHER); print("row 1 decider:", ROW1)

## 1. Smoke test: two of the shortest val questions, with cues

In [ ]:
!python scripts/run_decider_cache.py --samples data/processed/ovo_far_val.jsonl --cues {CUES} --limit 2 --shortest --set decider.variant=query_cues

## 2. Speed check

Should be close to row 1's 300 ms (the prompt is a little longer).

In [ ]:
import glob, json, statistics
rows = [json.loads(l) for f in glob.glob("results/cache/decider/*query_cues*/ovo_*.jsonl") for l in open(f) if l.strip()]
print(len(rows), "decisions; median", round(statistics.median(r["ms"] for r in rows)), "ms")

## 3. All val questions, regular 2 s ticks (resumable)

In [ ]:
!python scripts/run_decider_cache.py --samples data/processed/ovo_far_val.jsonl --cues {CUES} --set decider.variant=query_cues

## 4. Row 1 against row 2, same questions, same moments

For each question, how well p(Yes) separates moments near an event from the rest (AUC; 0.5 is no skill), averaged over questions,
and the change from adding the cues with a 95% interval that resamples whole videos. **A gain counts only if the interval excludes zero.**

In [ ]:
ROW2 = sorted(glob.glob("results/cache/decider/Qwen3-VL-2B-Instruct__query_cues__*"))[-1]
print("row 2 decider:", ROW2)
!python scripts/compare_deciders.py --a {ROW1} --b {ROW2} --samples data/processed/ovo_far_val.jsonl

## 5. Row 2 on its own, scored like row 1

Average F1 over thresholds and cooldowns. Compare with row 1's best (32.0, tuned on val) and the timer floors.

In [ ]:
!python scripts/replay_decider.py --cache {ROW2} --samples data/processed/ovo_far_val.jsonl

## 6. The decider with cues at the matcher's trigger moments (about 1,800 calls)

This redoes row 4 with the cues in the decider's prompt. It also fills the gaps where a trigger does not fall on a regular tick.

In [ ]:
!python scripts/run_gated_decider.py --samples data/processed/ovo_far_val.jsonl --matcher {MATCHER} --cues {CUES} --set decider.variant=query_cues

In [ ]:
!python scripts/compare_deciders.py --a {ROW1} --b {ROW2} --samples data/processed/ovo_far_val.jsonl --matcher {MATCHER}
!python scripts/replay_gated.py --matcher {MATCHER} --decider {ROW2} --samples data/processed/ovo_far_val.jsonl

## 7. Push the results

```
git add results/cache && git commit -m "Row 2 decider with cues, val" && git pull --rebase origin abhi && git push origin abhi
```